# DSAP-4 | Documentación y versionamiento del conjunto de datos

## Proyecto: Análisis de solicitudes ciudadanas NYC 311

### 1. Objetivo

Documentar la estructura, características, procedencia y
versionamiento del conjunto de datos utilizado en el proyecto DSA.

### 2. Conjunto de datos

- **Nombre:** NYC 311 Safety
- **Archivo:** nyc_311_safety.parquet
- **Formato:** Parquet
- **Fuente preliminar:** Sistema NYC 311
- **Licencia:** Pendiente de verificación
- **Versionamiento:** DVC con almacenamiento remoto en Amazon S3

### 3. Metodología

Se realizarán las siguientes actividades:

1. Carga y validación del conjunto de datos.
2. Identificación de dimensiones y tipos de variables.
3. Revisión de registros y categorías.
4. Identificación de cobertura temporal.
5. Evaluación preliminar de calidad.
6. Documentación del versionamiento mediante DVC.

Los resultados permitirán elaborar la ficha técnica del dataset
y generar evidencia reproducible para la primera entrega.

In [2]:
# DSAP-4 | Carga inicial del conjunto de datos

# 1. Importar las librerías necesarias
import pandas as pd
from pathlib import Path

# 2. Definir la ruta del dataset
# El notebook está en notebooks/ y los datos en data/raw/
ruta_dataset = Path("../data/raw/nyc_311_safety.parquet")

# 3. Comprobar que el archivo existe
if not ruta_dataset.exists():
    raise FileNotFoundError(
        f"No se encontró el dataset en: {ruta_dataset.resolve()}"
    )

# 4. Cargar el archivo Parquet en un DataFrame
df = pd.read_parquet(ruta_dataset)

# 5. Mostrar las dimensiones
print("Dataset cargado correctamente")
print(f"Número de registros: {df.shape[0]:,}")
print(f"Número de variables: {df.shape[1]}")

# 6. Mostrar los primeros cinco registros
print(df.head())

Dataset cargado correctamente
Número de registros: 3,167,279
Número de variables: 11
  dataType dataSubtype            dateTime            category  \
0   Safety     311_All 2023-12-01 04:57:07      HEAT/HOT WATER   
1   Safety     311_All 2023-02-24 18:25:45     Noise - Vehicle   
2   Safety     311_All 2023-08-03 16:11:03  Vendor Enforcement   
3   Safety     311_All 2023-09-27 11:07:32     Noise - Vehicle   
4   Safety     311_All 2023-08-05 18:46:29            Drinking   

       subcategory  status                 address   latitude  longitude  \
0  ENTIRE BUILDING  Closed    269 AMSTERDAM AVENUE  40.778972 -73.981351   
1  Car/Truck Music  Closed       1146 ELDER AVENUE  40.827250 -73.878852   
2      Food Vendor  Closed  1774 VICTORY BOULEVARD  40.613050 -74.122877   
3  Car/Truck Music  Closed      2189 JEROME AVENUE  40.856739 -73.904894   
4        In Public  Closed    2545 GRAND CONCOURSE  40.863451 -73.896389   

  source extendedProperties  
0    NaN                     
1

In [3]:
# DSAP-4 | Identificación del esquema del dataset

# Construir una tabla con la información de las columnas
esquema = pd.DataFrame({
    "Variable": df.columns,
    "Tipo de dato": df.dtypes.astype(str).values,
    "Valores no nulos": df.notna().sum().values,
    "Valores nulos": df.isna().sum().values
})

# Calcular porcentaje de datos faltantes
esquema["% Nulos"] = (
    esquema["Valores nulos"] / len(df) * 100
).round(2)

# Mostrar el esquema completo
display(esquema)

,Variable,Tipo de dato,Valores no nulos,Valores nulos,% Nulos
0,dataType,str,3167279,0,0.00
1,dataSubtype,str,3167279,0,0.00
2,dateTime,datetime64[ns],3167279,0,0.00
3,category,str,3167279,0,0.00
4,subcategory,str,3128519,38760,1.22
5,status,str,3167279,0,0.00
6,address,str,3045203,122076,3.85
7,latitude,float64,3118735,48544,1.53
8,longitude,float64,3118735,48544,1.53
9,source,str,0,3167279,100.00


¡Excelente! ✅ Ya tenemos una primera evaluación de la calidad del dataset, y encontramos un resultado importante que debemos documentar en el notebook.

## Paso 5 — Interpretar los valores faltantes

De los 3.167.279 registros, encontramos:

| Variable          | Valores nulos | % nulos |
| ----------------- | ------------- | ------- |
| `source`          | 3.167.279     | 100 %   |
| `address`         | 122.076       | 3,85 %  |
| `latitude`        | 48.544        | 1,53 %  |
| `longitude`       | 48.544        | 1,53 %  |
| `subcategory`     | 38.760        | 1,22 %  |
| Otras 6 variables | 0             | 0 %     |

### ¿Qué significan estos resultados?

- `source` requiere atención: no contiene ningún valor informado. Probablemente no aportará información para el análisis, aunque todavía no eliminaremos la columna.
- `address` presenta un 3,85 % de valores faltantes: debemos considerar esta limitación si realizamos análisis de direcciones.
- `latitude` y `longitude` tienen un 1,53 % de valores faltantes: algunos registros podrían no utilizarse en visualizaciones geográficas.
- `subcategory` tiene un 1,22 % de valores faltantes: debemos revisar si puede analizarse a partir de `category`.
- Las otras seis variables no presentan valores nulos detectados por Pandas.

## Paso 6 — Documentar los hallazgos dentro del notebook

Debajo de la tabla que acabas de generar, crea una celda de tipo Markdown y pega:

### 4.1. Interpretación del esquema y completitud

El conjunto de datos contiene **3.167.279 registros y 11 variables**, incluyendo información categórica, temporal y geográfica.

La evaluación inicial de completitud identifica los siguientes hallazgos:

- **Variable `source`:** presenta un 100 % de valores nulos, por lo que no aporta información observable en su estado actual. Se evaluará su exclusión en la etapa de limpieza.
- **Variable `address`:** presenta un 3,85 % de valores nulos, lo cual puede limitar los análisis que requieran direcciones.
- **Variables `latitude` y `longitude`:** presentan un 1,53 % de valores nulos cada una. Será necesario establecer criterios para tratar los registros sin coordenadas.
- **Variable `subcategory`:** presenta un 1,22 % de valores nulos, lo que puede afectar los análisis desagregados por subcategoría.
- **Variables restantes:** no presentan valores nulos según la evaluación realizada.

**Conclusión preliminar:** el dataset cuenta con información temporal, categórica y geográfica que permite plantear análisis descriptivos. Sin embargo, antes de su uso analítico, será necesario evaluar la consistencia de los registros, los valores duplicados y la calidad de las coordenadas.

En esta actividad se documentan los hallazgos, sin modificar el conjunto de datos original versionado mediante DVC.

Importante: estamos realizando DSAP-4, cuyo objetivo es documentar el dataset. Las decisiones de limpieza se implementarán posteriormente en DSAP-6.

Guarda el notebook con Ctrl + S y dime «Interpretación lista».

Después continuaremos con la cobertura temporal y las categorías del dataset, para completar su ficha técnica antes de preparar las evidencias en GitHub y Jira.

In [4]:
# DSAP-4 | Cobertura temporal del conjunto de datos

print("Fecha inicial:", df["dateTime"].min())
print("Fecha final:", df["dateTime"].max())

print("\nRegistros por año:")
display(
    df["dateTime"]
    .dt.year
    .value_counts()
    .sort_index()
    .rename_axis("Año")
    .to_frame("Registros")
)

Fecha inicial: 2023-01-01 00:00:00
Fecha final: 2023-12-24 12:00:00

Registros por año:


,Registros
Año,
2023,3167279


### 4.2. Cobertura temporal

El conjunto de datos contiene 3.167.279 registros
correspondientes al año 2023.

- **Fecha inicial:** 2023-01-01 00:00:00
- **Fecha final:** 2023-12-24 12:00:00
- **Años disponibles:** 2023

**Interpretación:**

El dataset permite estudiar patrones de solicitudes
ciudadanas durante el año 2023.

Para el componente predictivo será necesario considerar
que solo disponemos de un año de información, lo cual
limita el análisis de patrones interanuales.

Adicionalmente, se deberá verificar la continuidad diaria
de los registros antes de construir series temporales.

## 5. Fuente, procedencia y condiciones de uso

**Nombre del conjunto de datos local:** NYC 311 Safety.

**Archivo:** `nyc_311_safety.parquet`.

**Fuente pública de referencia:** NYC Open Data — 311 Service Requests from 2020 to Present.

**URL:** https://data.cityofnewyork.us/d/erm2-nwe9

**Proveedor de los datos originales:** NYC 311, Ciudad de Nueva York.

**Disponibilidad:** acceso público mediante el portal NYC Open Data.

**Licencia y condiciones de uso:** pendiente de confirmar las condiciones aplicables al conjunto original y a su transformación. El acceso público no se interpreta automáticamente como una licencia específica.

**Transformaciones:** el archivo local contiene 11 variables y registros del año 2023. Su estructura difiere del catálogo público de referencia, por lo que se requiere confirmar con el equipo el procedimiento de extracción, selección y transformación.

**Versionamiento:** el archivo está registrado mediante DVC y almacenado en Amazon S3.

**Estado de validación:** se verificaron las dimensiones, el esquema técnico, la completitud y la cobertura temporal del archivo local.